# 🏗️ PPE DETECTION & SAFETY MONITORING SYSTEM
## PHASE 02: YOLO11s MODEL DEVELOPMENT & EXPORT (STANDALONE)

Notebook này được thiết kế để **chạy 1 chạm (1-click run)** từ đầu đến cuối trên Google Colab T4 GPU:
1. Cài đặt môi trường cần thiết.
2. Tải và tự động chuẩn hóa 9 classes PPE chuẩn từ Roboflow.
3. Huấn luyện mô hình **YOLO11s (35 epochs ~ 5-7 phút trên T4 GPU)**.
4. Tự động đánh giá trên tập **Test** độc lập.
5. Export ra file **ONNX** cho backend/production.
6. **Tự động tải file zip trọng số về máy tính của bạn**.

### Bước 1: Cài đặt thư viện phụ thuộc

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime pyyaml matplotlib

import torch
import ultralytics
ultralytics.checks()

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Bước 2: Tải và chuẩn hóa tập dữ liệu 9 Classes

In [ ]:
import os
import glob
import yaml
import shutil
from roboflow import Roboflow

# 1. Tải raw dataset v30 từ Roboflow API
rf = Roboflow(api_key="FH8oooOj2IaJ4SlSjMtN")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(30)
raw_dataset = version.download("yolov11", location="/content/raw_data")

# 2. Chuẩn hóa sang 9 PPE classes
target_classes = [
    'Person', 'Hardhat', 'NO-Hardhat', 'Safety Vest', 
    'NO-Safety Vest', 'Mask', 'NO-Mask', 'Gloves', 'Safety Cone'
]

with open("/content/raw_data/data.yaml") as f:
    raw_names = yaml.safe_load(f)["names"]

raw_to_target = {raw_id: target_classes.index(name) for raw_id, name in enumerate(raw_names) if name in target_classes}

proc_dir = "/content/data_processed"
for s in ["train", "valid", "test"]:
    os.makedirs(f"{proc_dir}/{s}/images", exist_ok=True)
    os.makedirs(f"{proc_dir}/{s}/labels", exist_ok=True)
    for img_p in glob.glob(f"/content/raw_data/{s}/images/*.*"):
        stem = os.path.splitext(os.path.basename(img_p))[0]
        shutil.copy2(img_p, f"{proc_dir}/{s}/images/")
        lbl_p = f"/content/raw_data/{s}/labels/{stem}.txt"
        target_lines = []
        if os.path.exists(lbl_p):
            with open(lbl_p) as fp:
                for line in fp:
                    parts = line.strip().split()
                    if parts and int(parts[0]) in raw_to_target:
                        target_lines.append(f"{raw_to_target[int(parts[0])]} {' '.join(parts[1:])}")
        with open(f"{proc_dir}/{s}/labels/{stem}.txt", "w") as fp:
            if target_lines:
                fp.write("\n".join(target_lines) + "\n")

# 3. Ghi file data.yaml
data_cfg = {
    'path': proc_dir,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(target_classes),
    'names': target_classes
}
with open(f"{proc_dir}/data.yaml", "w") as f:
    yaml.dump(data_cfg, f, sort_keys=False)

print("✅ Dataset đã sẵn sàng tại:", f"{proc_dir}/data.yaml")

### Bước 3: Huấn luyện Mô hình YOLO11s (35 Epochs)

In [ ]:
from ultralytics import YOLO

print("🚀 Bắt đầu huấn luyện YOLO11s...")
model = YOLO("yolo11s.pt")

train_results = model.train(
    data="/content/data_processed/data.yaml",
    epochs=35,             # 35 epochs tối ưu (~6 phút trên T4 GPU)
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.002,
    cos_lr=True,
    mosaic=1.0,
    close_mosaic=8,
    project="/content/runs_ppe",
    name="yolo11s_train",
    exist_ok=True,
    plots=True
)

# Tự động bắt đúng đường dẫn weights không bao giờ bị sai
best_weight_path = str(train_results.save_dir / 'weights' / 'best.pt')
print("\n🎯 Trọng số tốt nhất đã được lưu tại:", best_weight_path)

### Bước 4: Đánh giá độc lập trên tập Test

In [ ]:
print("📊 Đánh giá kiểm định trên tập Test Set...")
best_model = YOLO(best_weight_path)
test_metrics = best_model.val(data="/content/data_processed/data.yaml", split="test")

print("\n=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===")
print(f"Precision (P)   : {test_metrics.box.mp:.4f}")
print(f"Recall (R)      : {test_metrics.box.mr:.4f}")
print(f"mAP@50          : {test_metrics.box.map50:.4f}")
print(f"mAP@50-95       : {test_metrics.box.map:.4f}")

### Bước 5: Export sang chuẩn ONNX và Tự Động Tải Về Máy

In [ ]:
from google.colab import files

# 1. Export ONNX
print("⚡ Đang export mô hình sang ONNX...")
export_onnx_path = best_model.export(format="onnx", dynamic=True, simplify=True)

# 2. Gom các file cần thiết
out_dir = "/content/ppe_models_output"
os.makedirs(out_dir, exist_ok=True)
shutil.copy2(best_weight_path, f"{out_dir}/ppe_best_yolo11.pt")
shutil.copy2(export_onnx_path, f"{out_dir}/ppe_best_yolo11.onnx")

# Copy thêm biểu đồ kết quả nếu có
for plot_f in ["results.png", "confusion_matrix.png"]:
    p = train_results.save_dir / plot_f
    if p.exists():
        shutil.copy2(str(p), f"{out_dir}/{plot_f}")

# 3. Nén zip và tự động tải về trình duyệt
!zip -r -j /content/ppe_models.zip /content/ppe_models_output/*

print("\n🎉 Đang bắt đầu tải file ppe_models.zip về máy tính của bạn...")
files.download("/content/ppe_models.zip")